In [ ]:
!pip install ucimlrepo

In [ ]:
from typing import dataclass_transform
import pandas as pd
from ucimlrepo import fetch_ucirepo


#read in datasets; example that follows is for the Pima Indians Diabetes Dataset; for the diabetes dataset use the \\
#code provided by UCI Ml to read in the datas
diabetes_df = pd.read_csv("diabetes.csv")
X_diab = diabetes_df.drop("Outcome", axis=1)
y_diab = diabetes_df["Outcome"]

# fetch dataset
glass_identification = fetch_ucirepo(id=42)

# data (as pandas dataframes)
X_glass = glass_identification.data.features
y_glass = glass_identification.data.targets

# metadata
print(glass_identification.metadata)

# variable information
print(glass_identification.variables)

print("Original Data:")
print(diabetes_df.head())

print("Original Data:")
print(pd.concat([X_glass, y_glass], axis=1).head())


# Display the classes and the no of samples in each type
print("Glass class distribution:")
print(y_glass.iloc[:, 0].value_counts().sort_index())

print("Diabetes class distribution:")
print(y_diab.value_counts().sort_index())

# Check missing values
print("Glass missing values:")
print(X_glass.isna().sum())

print("Diabetes missing values:")
print(X_diab.isna().sum())

# Replace impossible zeros with NaN (Diabetes only)
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
X_diab_fixed = X_diab.copy()
for col in zero_as_missing:
    if col in X_diab_fixed.columns:
        X_diab_fixed[col] = X_diab_fixed[col].replace(0, np.nan)

# set up the train and test splits
Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X_glass, y_glass.iloc[:, 0],
    test_size=0.30, random_state=42, stratify=y_glass.iloc[:, 0]
)

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_diab_fixed, y_diab,
    test_size=0.30, random_state=42, stratify=y_diab
)

# Impute missing values (median)
imp_g = SimpleImputer(strategy="median")
Xg_train_imp = pd.DataFrame(imp_g.fit_transform(Xg_train), columns=Xg_train.columns)
Xg_test_imp  = pd.DataFrame(imp_g.transform(Xg_test), columns=Xg_test.columns)

imp_d = SimpleImputer(strategy="median")
Xd_train_imp = pd.DataFrame(imp_d.fit_transform(Xd_train), columns=Xd_train.columns)
Xd_test_imp  = pd.DataFrame(imp_d.transform(Xd_test), columns=Xd_test.columns)

# Normalizing all feature values to lie in the range 0 to 1.
scaler_g = MinMaxScaler()
Xg_train_norm = pd.DataFrame(scaler_g.fit_transform(Xg_train_imp), columns=Xg_train_imp.columns)
Xg_test_norm  = pd.DataFrame(scaler_g.transform(Xg_test_imp), columns=Xg_test_imp.columns)

scaler_d = MinMaxScaler()
Xd_train_norm = pd.DataFrame(scaler_d.fit_transform(Xd_train_imp), columns=Xd_train_imp.columns)
Xd_test_norm  = pd.DataFrame(scaler_d.transform(Xd_test_imp), columns=Xd_test_imp.columns)

print("Glass normalized sample:")
print(Xg_train_norm.head())

print("Diabetes normalized sample:")
print(Xd_train_norm.head())

scaler_vis_d = MinMaxScaler()
X_diab_normalized_df = pd.DataFrame(
    scaler_vis_d.fit_transform(X_diab_fixed.fillna(X_diab_fixed.median(numeric_only=True))),
    columns=X_diab_fixed.columns
)

scaler_vis_g = MinMaxScaler()
X_glass_normalized_df = pd.DataFrame(
    scaler_vis_g.fit_transform(X_glass.fillna(X_glass.median(numeric_only=True))),
    columns=X_glass.columns
)


FileNotFoundError: [Errno 2] No such file or directory: 'diabetes.csv'

In [ ]:
# t-SNE Visualization before RBF Kernel
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
# Split features (X) and labels (y)
X = X_diab_normalized_df.values
y = y_diab.values

# Print the shapes of X and y
print("X shape:", X.shape)
print("y shape:", y.shape)

# Apply t-SNE to reduce dimensionality to 2 components (2D)
tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto")
X_tsne = tsne.fit_transform(X)

# Visualize the t-SNE results
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y, palette='icefire', legend="full")
plt.title("t-SNE Visualization Before RBF Kernel (Diabetes)")
plt.show()


In [ ]:
# t-SNE Visualization before RBF Kernel
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
# Split features (X) and labels (y)
X = X_glass_normalized_df.values
y = y_glass.iloc[:, 0].values

# Print the shapes of X and y
print("X shape:", X.shape)
print("y shape:", y.shape)

# Apply t-SNE to reduce dimensionality to 2 components (2D)
tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto")
X_tsne = tsne.fit_transform(X)

# Visualize the t-SNE results
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y, palette='icefire', legend="full")
plt.title("t-SNE Visualization Before RBF Kernel (Glass)")
plt.show()

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
# Apply RBF Kernel using sklearn.metrics.pairwise.rbf_kernel
X_rbf = rbf_kernel(X_diab_normalized_df.values)

# Apply t-SNE to visualize the class distributions after RBF Kernel
X_tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto").fit_transform(X_rbf)

# Visualize the t-SNE results
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y_diab.values, palette="icefire", legend="full")
plt.title("t-SNE Visualization After RBF Kernel Transformation (default gamma) - Diabetes")
plt.show()

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
# Apply RBF Kernel using sklearn.metrics.pairwise.rbf_kernel
X_rbf = rbf_kernel(X_glass_normalized_df.values)

# Apply t-SNE to visualize the class distributions after RBF Kernel
X_tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto").fit_transform(X_rbf)

# Visualize the t-SNE results
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y_glass.iloc[:, 0].values, palette="icefire", legend="full")
plt.title("t-SNE Visualization After RBF Kernel Transformation (default gamma) - Glass")
plt.show()


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
print("Linear Perceptron Before RBF Kernel (Diabetes)")
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import make_scorer, f1_score
from sklearn.linear_model import SGDClassifier
# set up the train and test splits

X_test = Xd_test_norm
y_test = yd_test

print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")
# Define the Perceptron model
model = SGDClassifier(loss="perceptron", random_state=42)

# Define the parameter grid for tuning the max_iter parameter
param_grid = {"max_iter": [100, 300, 500, 1000, 2000, 5000]}

# Define the Repeated K-Fold Cross-Validation with 10 folds and 3 repeats
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Use GridSearchCV to find the best parameters using cross-validation
scorer = make_scorer(f1_score, average="weighted")
grid = GridSearchCV(model, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)
grid.fit(Xd_train_norm, yd_train)

# Get the best parameter value
best_max_iter_diab_before = grid.best_params_["max_iter"]

print(f"Best max_iter: {best_max_iter_diab_before}")
# Get the best estimator (model) found by GridSearchCV
best_model_diab_before = grid.best_estimator_

# Cross-validation scores for accuracy
scores = cross_val_score(best_model_diab_before, Xd_train_norm, yd_train, scoring=scorer, cv=cv, n_jobs=-1)

# Calculate the weighted F1-score across all classes
weighted_f1_cv = np.mean(scores)
y_pred_test = best_model_diab_before.predict(X_test)
weighted_f1_test = f1_score(y_test, y_pred_test, average="weighted")

# Output the results
print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
print("Linear Perceptron Before RBF Kernel (Glass)")
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import make_scorer, f1_score
from sklearn.linear_model import SGDClassifier
# set up the train and test splits

X_test = Xg_test_norm
y_test = yg_test

print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")
# Define the Perceptron model
model = SGDClassifier(loss="perceptron", random_state=42)

# Define the parameter grid for tuning the max_iter parameter
param_grid = {"max_iter": [100, 300, 500, 1000, 2000, 5000]}

# Define the Repeated K-Fold Cross-Validation with 10 folds and 3 repeats
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Use GridSearchCV to find the best parameters using cross-validation
scorer = make_scorer(f1_score, average="weighted")
grid = GridSearchCV(model, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)
grid.fit(Xg_train_norm, yg_train)

# Get the best parameter value
best_max_iter_glass_before = grid.best_params_["max_iter"]

print(f"Best max_iter: {best_max_iter_glass_before}")
# Get the best estimator (model) found by GridSearchCV
best_model_glass_before = grid.best_estimator_

# Cross-validation scores for accuracy
scores = cross_val_score(best_model_glass_before, Xg_train_norm, yg_train, scoring=scorer, cv=cv, n_jobs=-1)

# Calculate the weighted F1-score across all classes
weighted_f1_cv = np.mean(scores)
y_pred_test = best_model_glass_before.predict(X_test)
weighted_f1_test = f1_score(y_test, y_pred_test, average="weighted")

# Output the results
print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import make_scorer, f1_score
from sklearn.linear_model import SGDClassifier
import numpy as np
print("Linear Perceptron After RBF Kernel (Diabetes)")
# Apply the RBF kernel transformation
gamma_default = 1.0 / Xd_train_norm.shape[1]
X_train_rbf = rbf_kernel(Xd_train_norm.values, Xd_train_norm.values, gamma=gamma_default)

# Define the Linear Perceptron model
rbf_li_perceptron_2 = SGDClassifier(loss="perceptron", random_state=42)
# Define the parameter grid for tuning the max_iter parameter
param_grid = {"max_iter": [100, 300, 500, 1000, 2000, 5000]}

# Define the Repeated K-Fold Cross-Validation with 10 folds and 3 repeats
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Use GridSearchCV to find the best parameters using cross-validation
scorer = make_scorer(f1_score, average="weighted")
grid = GridSearchCV(rbf_li_perceptron_2, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)

# Fit the model on the RBF transformed data
grid.fit(X_train_rbf, yd_train)

# Get the best parameters
best_max_iter_diab_rbf = grid.best_params_["max_iter"]

print(f"Best max_iter: {best_max_iter_diab_rbf}")
# Get the best estimator (model) found by GridSearchCV
best_model_diab_rbf = grid.best_estimator_

# Calculate the weighted F1-score across all classes
scores = cross_val_score(best_model_diab_rbf, X_train_rbf, yd_train, scoring=scorer, cv=cv, n_jobs=-1)
weighted_f1_cv = np.mean(scores)

X_test_rbf = rbf_kernel(Xd_test_norm.values, Xd_train_norm.values, gamma=gamma_default)
y_pred_test = best_model_diab_rbf.predict(X_test_rbf)
weighted_f1_test = f1_score(yd_test, y_pred_test, average="weighted")

print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)


In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import make_scorer, f1_score
from sklearn.linear_model import SGDClassifier
import numpy as np
print("Linear Perceptron After RBF Kernel (Glass)")
# Apply the RBF kernel transformation
gamma_default = 1.0 / Xg_train_norm.shape[1]
X_train_rbf = rbf_kernel(Xg_train_norm.values, gamma=gamma_default)

# Define the Linear Perceptron model
rbf_li_perceptron_2 = SGDClassifier(loss="perceptron", random_state=42)
# Define the parameter grid for tuning the max_iter parameter
param_grid = {"max_iter": [100, 300, 500, 1000, 2000, 5000]}

# Define the Repeated K-Fold Cross-Validation with 10 folds and 3 repeats
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Use GridSearchCV to find the best parameters using cross-validation
scorer = make_scorer(f1_score, average="weighted")
grid = GridSearchCV(rbf_li_perceptron_2, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)

# Fit the model on the RBF transformed data
grid.fit(X_train_rbf, yg_train)

# Get the best parameters
best_max_iter_glass_rbf = grid.best_params_["max_iter"]

print(f"Best max_iter: {best_max_iter_glass_rbf}")
# Get the best estimator (model) found by GridSearchCV
best_model_glass_rbf = grid.best_estimator_

# Calculate the weighted F1-score across all classes
scores = cross_val_score(best_model_glass_rbf, X_train_rbf, yg_train, scoring=scorer, cv=cv, n_jobs=-1)
weighted_f1_cv = np.mean(scores)

X_test_rbf = rbf_kernel(Xg_test_norm.values, Xg_train_norm.values, gamma=gamma_default)
y_pred_test = best_model_glass_rbf.predict(X_test_rbf)
weighted_f1_test = f1_score(yg_test, y_pred_test, average="weighted")

print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)


In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import make_scorer, f1_score
import numpy as np

# Apply RBF Kernel using sklearn.metrics.pairwise.rbf_kernel
X_rbf = rbf_kernel(X_diab_normalized_df.values, gamma=1.0 / Xd_train_norm.shape[1])

# Print the shapes of X_rbf
print("Initial X_rbf shape:", X_rbf.shape)

# Apply t-SNE to visualize the class distributions after RBF Kernel
X_tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto").fit_transform(X_rbf)

# Visualize the t-SNE results
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y_diab.values, palette="icefire", legend="full")
plt.title("t-SNE Visualization After RBF Kernel Transformation (default gamma) - Diabetes")
plt.show()

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scorer = make_scorer(f1_score, average="weighted")

gammas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
best_gamma_diab = None
best_score = -1

for g in gammas:
    X_train_rbf = rbf_kernel(Xd_train_norm.values, gamma=g)
    m = SGDClassifier(loss="perceptron", random_state=42, max_iter=best_max_iter_diab_rbf)
    s = cross_val_score(m, X_train_rbf, yd_train, scoring=scorer, cv=cv, n_jobs=-1)
    if np.mean(s) > best_score:
        best_score = np.mean(s)
        best_gamma_diab = g

X_rbf = rbf_kernel(X_diab_normalized_df.values, gamma=best_gamma_diab)
print("Tuned X_rbf shape:", X_rbf.shape)

X_tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto").fit_transform(X_rbf)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y_diab.values, palette="icefire", legend="full")
plt.title("t-SNE Visualization After RBF Kernel Transformation (tuned gamma) - Diabetes")
plt.show()

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import make_scorer, f1_score
import numpy as np

# Apply RBF Kernel using sklearn.metrics.pairwise.rbf_kernel
X_rbf = rbf_kernel(X_glass_normalized_df.values, gamma=1.0 / Xg_train_norm.shape[1])

# Print the shapes of X_rbf
print("Initial X_rbf shape:", X_rbf.shape)

# Apply t-SNE to visualize the class distributions after RBF Kernel
X_tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto").fit_transform(X_rbf)

# Visualize the t-SNE results
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y_glass.iloc[:, 0].values, palette="icefire", legend="full")
plt.title("t-SNE Visualization After RBF Kernel Transformation (default gamma) - Glass")
plt.show()

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scorer = make_scorer(f1_score, average="weighted")

gammas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
best_gamma_glass = None
best_score = -1

for g in gammas:
    X_train_rbf = rbf_kernel(Xg_train_norm.values, gamma=g)
    m = SGDClassifier(loss="perceptron", random_state=42, max_iter=best_max_iter_glass_rbf)
    s = cross_val_score(m, X_train_rbf, yg_train, scoring=scorer, cv=cv, n_jobs=-1)
    if np.mean(s) > best_score:
        best_score = np.mean(s)
        best_gamma_glass = g

X_rbf = rbf_kernel(X_glass_normalized_df.values, gamma=best_gamma_glass)
print("Tuned X_rbf shape:", X_rbf.shape)

X_tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto").fit_transform(X_rbf)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y_glass.iloc[:, 0].values, palette="icefire", legend="full")
plt.title("t-SNE Visualization After RBF Kernel Transformation (tuned gamma) - Glass")
plt.show()


print("Logistic Regression Before RBF Kernel (Diabetes)")
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scorer = make_scorer(f1_score, average="weighted")

model = SGDClassifier(loss="log_loss", random_state=42, class_weight=None)
param_grid = {"max_iter": [100, 300, 500, 1000, 2000, 5000]}

grid = GridSearchCV(model, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)
grid.fit(Xd_train_norm, yd_train)

best_max_iter_log_diab_before = grid.best_params_["max_iter"]
print("Best max_iter:", best_max_iter_log_diab_before)

best_log_diab_before = grid.best_estimator_
scores = cross_val_score(best_log_diab_before, Xd_train_norm, yd_train, scoring=scorer, cv=cv, n_jobs=-1)
weighted_f1_cv = np.mean(scores)

y_pred_test = best_log_diab_before.predict(Xd_test_norm)
weighted_f1_test = f1_score(yd_test, y_pred_test, average="weighted")

print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)

In [ ]:
print("Logistic Regression After RBF Kernel (Diabetes)")
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scorer = make_scorer(f1_score, average="weighted")

gamma_default = 1.0 / Xd_train_norm.shape[1]
X_train_rbf = rbf_kernel(Xd_train_norm.values, Xd_train_norm.values, gamma=gamma_default)

model = SGDClassifier(loss="log_loss", random_state=42, class_weight=None)
param_grid = {"max_iter": [100, 300, 500, 1000, 2000, 5000]}

grid = GridSearchCV(model, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)
grid.fit(X_train_rbf, yd_train)

best_max_iter_log_diab_rbf = grid.best_params_["max_iter"]
print("Best max_iter:", best_max_iter_log_diab_rbf)

best_log_diab_rbf = grid.best_estimator_
scores = cross_val_score(best_log_diab_rbf, X_train_rbf, yd_train, scoring=scorer, cv=cv, n_jobs=-1)
weighted_f1_cv = np.mean(scores)

X_test_rbf = rbf_kernel(Xd_test_norm.values, Xd_train_norm.values, gamma=gamma_default)
y_pred_test = best_log_diab_rbf.predict(X_test_rbf)
weighted_f1_test = f1_score(yd_test, y_pred_test, average="weighted")

print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)

print("Tuning gamma for Logistic Regression (Diabetes)")
gammas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
best_gamma_log_diab = None
best_score = -1

for g in gammas:
    X_train_rbf_g = rbf_kernel(Xd_train_norm.values, Xd_train_norm.values, gamma=g)
    m = SGDClassifier(loss="log_loss", random_state=42, class_weight=None, max_iter=best_max_iter_log_diab_rbf)
    s = cross_val_score(m, X_train_rbf_g, yd_train, scoring=scorer, cv=cv, n_jobs=-1)
    if np.mean(s) > best_score:
        best_score = np.mean(s)
        best_gamma_log_diab = g

print("Best gamma:", best_gamma_log_diab)
print("Best gamma CV F1:", best_score)


In [ ]:
print("Logistic Regression Before RBF Kernel (Glass)")
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scorer = make_scorer(f1_score, average="weighted")

model = SGDClassifier(loss="log_loss", random_state=42, class_weight=None)
param_grid = {"max_iter": [100, 300, 500, 1000, 2000, 5000]}

grid = GridSearchCV(model, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)
grid.fit(Xg_train_norm, yg_train)

best_max_iter_log_glass_before = grid.best_params_["max_iter"]
print("Best max_iter:", best_max_iter_log_glass_before)

best_log_glass_before = grid.best_estimator_
scores = cross_val_score(best_log_glass_before, Xg_train_norm, yg_train, scoring=scorer, cv=cv, n_jobs=-1)
weighted_f1_cv = np.mean(scores)

y_pred_test = best_log_glass_before.predict(Xg_test_norm)
weighted_f1_test = f1_score(yg_test, y_pred_test, average="weighted")

print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)


In [ ]:
print("Logistic Regression After RBF Kernel (Glass)")
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scorer = make_scorer(f1_score, average="weighted")

gamma_default = 1.0 / Xg_train_norm.shape[1]
X_train_rbf = rbf_kernel(Xg_train_norm.values, Xg_train_norm.values, gamma=gamma_default)

model = SGDClassifier(loss="log_loss", random_state=42, class_weight=None)
param_grid = {"max_iter": [100, 300, 500, 1000, 2000, 5000]}

grid = GridSearchCV(model, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)
grid.fit(X_train_rbf, yg_train)

best_max_iter_log_glass_rbf = grid.best_params_["max_iter"]
print("Best max_iter:", best_max_iter_log_glass_rbf)

best_log_glass_rbf = grid.best_estimator_
scores = cross_val_score(best_log_glass_rbf, X_train_rbf, yg_train, scoring=scorer, cv=cv, n_jobs=-1)
weighted_f1_cv = np.mean(scores)

X_test_rbf = rbf_kernel(Xg_test_norm.values, Xg_train_norm.values, gamma=gamma_default)
y_pred_test = best_log_glass_rbf.predict(X_test_rbf)
weighted_f1_test = f1_score(yg_test, y_pred_test, average="weighted")

print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)

print("Tuning gamma for Logistic Regression (Glass)")
gammas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
best_gamma_log_glass = None
best_score = -1

for g in gammas:
    X_train_rbf_g = rbf_kernel(Xg_train_norm.values, Xg_train_norm.values, gamma=g)
    m = SGDClassifier(loss="log_loss", random_state=42, class_weight=None, max_iter=best_max_iter_log_glass_rbf)
    s = cross_val_score(m, X_train_rbf_g, yg_train, scoring=scorer, cv=cv, n_jobs=-1)
    if np.mean(s) > best_score:
        best_score = np.mean(s)
        best_gamma_log_glass = g

print("Best gamma:", best_gamma_log_glass)
print("Best gamma CV F1:", best_score)

In [ ]:
print("MLPClassifier (Diabetes)")
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scorer = make_scorer(f1_score, average="weighted")

mlp = MLPClassifier(random_state=42)
param_grid = {
    "max_iter": [200, 500, 1000, 2000],
    "n_iter_no_change": [5, 10, 20]
}

grid = GridSearchCV(mlp, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)
grid.fit(Xd_train_norm, yd_train)

print("Best params:", grid.best_params_)
best_mlp_diab = grid.best_estimator_

scores = cross_val_score(best_mlp_diab, Xd_train_norm, yd_train, scoring=scorer, cv=cv, n_jobs=-1)
weighted_f1_cv = np.mean(scores)

y_pred_test = best_mlp_diab.predict(Xd_test_norm)
weighted_f1_test = f1_score(yd_test, y_pred_test, average="weighted")

print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)

In [ ]:
print("MLPClassifier (Glass)")
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scorer = make_scorer(f1_score, average="weighted")

mlp = MLPClassifier(random_state=42)
param_grid = {
    "max_iter": [200, 500, 1000, 2000],
    "n_iter_no_change": [5, 10, 20]
}

grid = GridSearchCV(mlp, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1)
grid.fit(Xg_train_norm, yg_train)

print("Best params:", grid.best_params_)
best_mlp_glass = grid.best_estimator_

scores = cross_val_score(best_mlp_glass, Xg_train_norm, yg_train, scoring=scorer, cv=cv, n_jobs=-1)
weighted_f1_cv = np.mean(scores)

y_pred_test = best_mlp_glass.predict(Xg_test_norm)
weighted_f1_test = f1_score(yg_test, y_pred_test, average="weighted")

print("Cross Validation Scores: ", scores)
print("Average CV Score: ", weighted_f1_cv)
print("Test Weighted F1: ", weighted_f1_test)

In [ ]:
print("Logistic Regression Feature Significance (Diabetes)")
model = SGDClassifier(loss="log_loss", random_state=42, class_weight=None, max_iter=5000)
model.fit(Xd_train_norm, yd_train)

coefs = model.coef_
importance = np.mean(np.abs(coefs), axis=0)

feature_names = Xd_train_norm.columns.tolist()
ranked = sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True)

top5 = ranked[:5]
bottom5 = ranked[-5:]

print("Top 5 predictors:", top5)
print("Bottom 5 predictors:", bottom5)

drop_features = [name for name, _ in bottom5]
Xd_train_drop = Xd_train_norm.drop(columns=drop_features)
Xd_test_drop = Xd_test_norm.drop(columns=drop_features)

model2 = SGDClassifier(loss="log_loss", random_state=42, class_weight=None, max_iter=5000)
model2.fit(Xd_train_drop, yd_train)

pred = model2.predict(Xd_test_drop)
f1 = f1_score(yd_test, pred, average="weighted")
print("Weighted F1 after dropping 5 least significant:", f1)


In [ ]:
print("Logistic Regression Feature Significance (Glass)")
model = SGDClassifier(loss="log_loss", random_state=42, class_weight=None, max_iter=5000)
model.fit(Xg_train_norm, yg_train)

coefs = model.coef_
importance = np.mean(np.abs(coefs), axis=0)

feature_names = Xg_train_norm.columns.tolist()
ranked = sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True)

top5 = ranked[:5]
bottom5 = ranked[-5:]

print("Top 5 predictors:", top5)
print("Bottom 5 predictors:", bottom5)

drop_features = [name for name, _ in bottom5]
Xg_train_drop = Xg_train_norm.drop(columns=drop_features)
Xg_test_drop = Xg_test_norm.drop(columns=drop_features)

model2 = SGDClassifier(loss="log_loss", random_state=42, class_weight=None, max_iter=5000)
model2.fit(Xg_train_drop, yg_train)

pred = model2.predict(Xg_test_drop)
f1 = f1_score(yg_test, pred, average="weighted")
print("Weighted F1 after dropping 5 least significant:", f1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsRegressor # Changed to Regressor

# 3. Train classifier on ORIGINAL features
clf = SGDClassifier(loss="perceptron", random_state=42, max_iter=best_max_iter_diab_before)
clf.fit(Xd_train_norm.values, yd_train.values)
print(f"Shape of X_true (features): {Xd_train_norm.values.shape}")

# 4. Apply t-SNE for visualization only
tsne = TSNE(n_components=2, random_state=42, perplexity=30, learning_rate=200)
X_tsne = tsne.fit_transform(Xd_train_norm.values)

# 5. Create mesh grid in t-SNE space
x_min, x_max = X_tsne[:, 0].min() - 1, X_tsne[:, 0].max() + 1
y_min, y_max = X_tsne[:, 1].min() - 1, X_tsne[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500),
                     np.linspace(y_min, y_max, 500))

# 6. Map mesh grid points back to original feature space
#    This is tricky because t-SNE is non-invertible.
#    Instead, we approximate decision boundaries by predicting nearest neighbor's original feature vector.

# Fit a KNN regressor to map t-SNE space -> original features
knn_mapper = KNeighborsRegressor(n_neighbors=1) # Changed to Regressor
knn_mapper.fit(X_tsne, Xd_train_norm.values)

# Map each mesh point to nearest original feature vector
mesh_points_original = knn_mapper.predict(np.c_[xx.ravel(), yy.ravel()])

# Predict labels using the classifier trained on original features
Z = clf.predict(mesh_points_original)
Z = Z.reshape(xx.shape)

# 7. Plot decision boundary and t-SNE points
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=yd_train.values, edgecolor='k', cmap=plt.cm.coolwarm)
plt.legend(handles=scatter.legend_elements()[0], labels=list(np.unique(yd_train.values))) # Converted to list
plt.title("t-SNE Visualization with Classification Boundary (Trained on Original Features) - Diabetes")
plt.xlabel("t-SNE Feature 1")
plt.ylabel("t-SNE Feature 2")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsRegressor # Changed to Regressor

# 3. Train classifier on ORIGINAL features
clf = SGDClassifier(loss="perceptron", random_state=42, max_iter=best_max_iter_glass_before)
clf.fit(Xg_train_norm.values, yg_train.values)
print(f"Shape of X_true (features): {Xg_train_norm.values.shape}")

# 4. Apply t-SNE for visualization only
tsne = TSNE(n_components=2, random_state=42, perplexity=30, learning_rate=200)
X_tsne = tsne.fit_transform(Xg_train_norm.values)

# 5. Create mesh grid in t-SNE space
x_min, x_max = X_tsne[:, 0].min() - 1, X_tsne[:, 0].max() + 1
y_min, y_max = X_tsne[:, 1].min() - 1, X_tsne[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500),
                     np.linspace(y_min, y_max, 500))

# 6. Map mesh grid points back to original feature space
#    This is tricky because t-SNE is non-invertible.
#    Instead, we approximate decision boundaries by predicting nearest neighbor's original feature vector.

# Fit a KNN regressor to map t-SNE space -> original features
knn_mapper = KNeighborsRegressor(n_neighbors=1) # Changed to Regressor
knn_mapper.fit(X_tsne, Xg_train_norm.values)

# Map each mesh point to nearest original feature vector
mesh_points_original = knn_mapper.predict(np.c_[xx.ravel(), yy.ravel()])

# Predict labels using the classifier trained on original features
Z = clf.predict(mesh_points_original)
Z = Z.reshape(xx.shape)

# 7. Plot decision boundary and t-SNE points
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=yg_train.values, edgecolor='k', cmap=plt.cm.coolwarm)
plt.legend(handles=scatter.legend_elements()[0], labels=list(np.unique(yg_train.values))) # Converted to list
plt.title("t-SNE Visualization with Classification Boundary (Trained on Original Features) - Glass")
plt.xlabel("t-SNE Feature 1")
plt.ylabel("t-SNE Feature 2")
plt.show()
